<a href="https://colab.research.google.com/github/peterbabulik/TalkToArgon10100102026/blob/main/Argon_Chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import datetime
import html
import json
import os
from pathlib import Path
import re
import shlex
import traceback

from IPython import get_ipython
from IPython.display import display
from IPython.utils.capture import capture_output
import ipywidgets as widgets
import kaggle_benchmarks as kbench


# Optional Markdown support.
try:
    import markdown

    HAS_MARKDOWN = True
except ImportError:
    HAS_MARKDOWN = False


# Builds a markdown code fence programmatically so the source
# contains no literal triple backticks.
CODE_FENCE = "`" * 3

# Attachment limits and language mapping.
ATTACH_MAX_BYTES = 200_000
ATTACH_MAX_FILES = 10
TEXT_LANG_BY_EXT = {
    ".py": "python",
    ".js": "javascript",
    ".ts": "typescript",
    ".html": "html",
    ".css": "css",
    ".sh": "bash",
    ".sql": "sql",
    ".json": "json",
    ".yaml": "yaml",
    ".yml": "yaml",
    ".md": "markdown",
    ".cpp": "cpp",
    ".c": "c",
    ".java": "java",
    ".csv": "text",
    ".txt": "text",
}


class KaggleLLMWorkbench:
    def __init__(self, work_dir="/kaggle/working"):
        self.work_dir = Path(work_dir)
        self.work_dir.mkdir(parents=True, exist_ok=True)

        self.chat_history = []
        self.attachments = []
        self._last_exec_log = ""

        self.available_models = list(kbench.llms.keys())

        if not self.available_models:
            raise RuntimeError(
                "No language models were found in kaggle_benchmarks.llms."
            )

        self.default_model = (
            "google/gemini-4-argon"
            if "google/gemini-4-argon" in self.available_models
            else self.available_models[0]
        )

        self._init_ui()
        self._bind_events()

    # ==========================================
    # 1. UI INITIALIZATION
    # ==========================================
    def _init_ui(self):
        self.theme = widgets.HTML(
            value="""
            <style>
                :root {
                    --kwb-surface: #ffffff;
                    --kwb-surface-soft: #f8fafc;
                    --kwb-border: #dce4ee;
                    --kwb-border-strong: #cbd5e1;
                    --kwb-text: #172033;
                    --kwb-muted: #64748b;
                    --kwb-primary: #5856d6;
                    --kwb-code-bg: #111827;
                    --kwb-radius: 14px;
                    --kwb-shadow: 0 10px 28px rgba(15, 23, 42, 0.08);
                }

                .kwb-root {
                    width: 100%;
                    box-sizing: border-box;
                    color: var(--kwb-text);
                    font-family:
                        Inter,
                        ui-sans-serif,
                        system-ui,
                        -apple-system,
                        BlinkMacSystemFont,
                        "Segoe UI",
                        sans-serif;
                }

                .kwb-shell {
                    gap: 13px;
                }

                .kwb-header {
                    box-sizing: border-box;
                    position: relative;
                    overflow: hidden;
                    padding: 23px 25px;
                    border-radius: 18px;
                    background:
                        radial-gradient(
                            circle at 88% 8%,
                            rgba(255, 255, 255, 0.18),
                            transparent 27%
                        ),
                        linear-gradient(
                            135deg,
                            #30327a 0%,
                            #5856d6 55%,
                            #7c3aed 100%
                        );
                    box-shadow: 0 16px 38px rgba(50, 51, 123, 0.22);
                    color: #ffffff;
                }

                .kwb-header-row {
                    display: flex;
                    justify-content: space-between;
                    align-items: center;
                    gap: 18px;
                    flex-wrap: wrap;
                }

                .kwb-brand {
                    display: flex;
                    align-items: center;
                    gap: 14px;
                }

                .kwb-logo {
                    display: grid;
                    place-items: center;
                    width: 49px;
                    height: 49px;
                    flex: 0 0 49px;
                    border: 1px solid rgba(255, 255, 255, 0.24);
                    border-radius: 14px;
                    background: rgba(255, 255, 255, 0.13);
                    box-shadow:
                        inset 0 1px 0 rgba(255, 255, 255, 0.18),
                        0 8px 20px rgba(24, 24, 70, 0.18);
                    font-size: 25px;
                }

                .kwb-title {
                    font-size: 22px;
                    line-height: 1.2;
                    font-weight: 760;
                    letter-spacing: -0.02em;
                }

                .kwb-subtitle {
                    max-width: 620px;
                    margin-top: 5px;
                    color: rgba(255, 255, 255, 0.78);
                    font-size: 12px;
                    line-height: 1.5;
                }

                .kwb-header-badges {
                    display: flex;
                    align-items: center;
                    gap: 7px;
                    flex-wrap: wrap;
                }

                .kwb-header-badge {
                    padding: 7px 10px;
                    border: 1px solid rgba(255, 255, 255, 0.20);
                    border-radius: 999px;
                    background: rgba(255, 255, 255, 0.12);
                    color: #ffffff;
                    font-size: 10.5px;
                    font-weight: 650;
                    backdrop-filter: blur(8px);
                }

                .kwb-panel {
                    box-sizing: border-box;
                    padding: 15px;
                    border: 1px solid var(--kwb-border);
                    border-radius: var(--kwb-radius);
                    background: var(--kwb-surface);
                    box-shadow: var(--kwb-shadow);
                }

                .kwb-section-title {
                    color: var(--kwb-text);
                    font-size: 14px;
                    font-weight: 750;
                }

                .kwb-section-subtitle {
                    margin-top: 3px;
                    color: var(--kwb-muted);
                    font-size: 11px;
                    line-height: 1.45;
                }

                .kwb-toolbar {
                    align-items: center;
                    flex-wrap: wrap;
                }

                .kwb-toolbar-label {
                    color: var(--kwb-muted);
                    font-size: 10.5px;
                    font-weight: 750;
                    letter-spacing: 0.08em;
                    text-transform: uppercase;
                }

                .kwb-root .widget-label {
                    color: var(--kwb-text);
                    font-weight: 650;
                }

                .kwb-root select,
                .kwb-root textarea,
                .kwb-root input[type="text"],
                .kwb-root input[type="number"] {
                    box-sizing: border-box;
                    border: 1px solid var(--kwb-border-strong) !important;
                    border-radius: 10px !important;
                    background: var(--kwb-surface) !important;
                    color: var(--kwb-text) !important;
                    outline: none;
                    transition:
                        border-color 150ms ease,
                        box-shadow 150ms ease;
                }

                .kwb-root select:focus,
                .kwb-root textarea:focus,
                .kwb-root input:focus {
                    border-color: var(--kwb-primary) !important;
                    box-shadow:
                        0 0 0 3px rgba(88, 86, 214, 0.14) !important;
                }

                .kwb-prompt textarea {
                    min-height: 160px;
                    padding: 15px 16px !important;
                    border-color: #cfd8e5 !important;
                    border-radius: 13px !important;
                    background: #ffffff !important;
                    color: #172033 !important;
                    font-family:
                        ui-monospace,
                        SFMono-Regular,
                        Menlo,
                        Monaco,
                        Consolas,
                        monospace;
                    font-size: 13px;
                    line-height: 1.55;
                    resize: vertical;
                    box-shadow:
                        inset 0 1px 2px rgba(15, 23, 42, 0.03);
                }

                .kwb-system-prompt textarea {
                    padding: 12px 13px !important;
                    font-family:
                        ui-monospace,
                        SFMono-Regular,
                        Menlo,
                        Monaco,
                        Consolas,
                        monospace;
                    font-size: 12px;
                    line-height: 1.5;
                    resize: vertical;
                }

                .kwb-root .jupyter-widgets.widget-accordion {
                    overflow: hidden;
                    border: 1px solid var(--kwb-border);
                    border-radius: var(--kwb-radius);
                    background: var(--kwb-surface);
                    box-shadow: 0 6px 20px rgba(15, 23, 42, 0.05);
                }

                .kwb-settings-content {
                    padding: 7px 4px 5px;
                }

                .kwb-actionbar {
                    align-items: center;
                    flex-wrap: wrap;
                }

                .kwb-status-wrap {
                    min-width: 230px;
                    margin-left: auto;
                }

                .kwb-status {
                    display: inline-flex;
                    align-items: center;
                    min-height: 31px;
                    box-sizing: border-box;
                    padding: 6px 11px;
                    border: 1px solid var(--kwb-border);
                    border-radius: 999px;
                    background: var(--kwb-surface-soft);
                    font-size: 12px;
                    line-height: 1.35;
                    font-weight: 600;
                }

                .kwb-root .widget-button {
                    border: 0 !important;
                    border-radius: 9px !important;
                    font-weight: 700 !important;
                    box-shadow: 0 2px 5px rgba(15, 23, 42, 0.12);
                    transition:
                        transform 130ms ease,
                        box-shadow 130ms ease,
                        filter 130ms ease;
                }

                .kwb-root .widget-button:hover:not(:disabled) {
                    transform: translateY(-1px);
                    filter: brightness(1.03);
                    box-shadow: 0 6px 14px rgba(15, 23, 42, 0.15);
                }

                .kwb-root .widget-button:active:not(:disabled) {
                    transform: translateY(0);
                }

                .kwb-root .widget-button:disabled {
                    opacity: 0.55;
                    box-shadow: none;
                }

                .kwb-root .jupyter-button.mod-primary {
                    background:
                        linear-gradient(135deg, #5856d6, #7048d7);
                    color: #ffffff;
                }

                .kwb-root .jupyter-button.mod-success {
                    background:
                        linear-gradient(135deg, #12815b, #19a373);
                    color: #ffffff;
                }

                .kwb-root .jupyter-button.mod-danger {
                    background:
                        linear-gradient(135deg, #c73547, #e34e5c);
                    color: #ffffff;
                }

                .kwb-root .jupyter-button.mod-warning {
                    background:
                        linear-gradient(135deg, #dc811b, #f0a02d);
                    color: #ffffff;
                }

                .kwb-root .jupyter-button.mod-info {
                    background:
                        linear-gradient(135deg, #2871c7, #4293e3);
                    color: #ffffff;
                }

                .kwb-generated-files {
                    padding: 11px 13px;
                    border: 1px solid #cfdcff;
                    border-radius: 11px;
                    background: #f2f6ff;
                    color: #334155;
                    font-size: 12px;
                    line-height: 1.75;
                }

                .kwb-generated-files strong {
                    color: #315db5;
                }

                .kwb-generated-files a {
                    color: #4057c8;
                    font-weight: 650;
                    text-decoration: none;
                }

                .kwb-generated-files a:hover {
                    text-decoration: underline;
                }

                .kwb-console {
                    box-sizing: border-box;
                    padding: 15px;
                    border: 1px solid #334155;
                    border-radius: var(--kwb-radius);
                    background: #111827;
                    box-shadow: 0 12px 30px rgba(15, 23, 42, 0.2);
                }

                .kwb-console-title {
                    color: #e5e7eb;
                    font-size: 11px;
                    font-weight: 750;
                    letter-spacing: 0.06em;
                    text-transform: uppercase;
                }

                .kwb-console-output {
                    box-sizing: border-box;
                    border: 1px solid #374151 !important;
                    border-radius: 10px;
                    background: #0b1220 !important;
                    color: #d7e0ec;
                }

                .kwb-chat-shell {
                    box-sizing: border-box;
                    border: 1px solid var(--kwb-border);
                    border-radius: 16px;
                    background:
                        linear-gradient(
                            180deg,
                            rgba(248, 250, 252, 0.97),
                            rgba(243, 246, 251, 0.97)
                        );
                    box-shadow: var(--kwb-shadow);
                }

                .kwb-chat-scroll {
                    scrollbar-width: thin;
                    scrollbar-color: #aeb9c8 transparent;
                }

                .kwb-chat-scroll::-webkit-scrollbar {
                    width: 8px;
                }

                .kwb-chat-scroll::-webkit-scrollbar-thumb {
                    border-radius: 999px;
                    background: #aeb9c8;
                }

                .kwb-chat-scroll::-webkit-scrollbar-track {
                    background: transparent;
                }

                .kwb-chat-card {
                    box-sizing: border-box;
                    margin: 12px 2px;
                    border-radius: 13px;
                    animation: kwb-card-in 180ms ease-out;
                }

                @keyframes kwb-card-in {
                    from {
                        opacity: 0;
                        transform: translateY(4px);
                    }
                    to {
                        opacity: 1;
                        transform: translateY(0);
                    }
                }

                .kwb-user-card {
                    margin-left: 7%;
                    padding: 13px 15px;
                    border: 1px solid #cddcff;
                    background:
                        linear-gradient(135deg, #edf4ff, #f7faff);
                    box-shadow:
                        0 5px 14px rgba(41, 112, 214, 0.07);
                }

                .kwb-model-card {
                    margin-right: 3%;
                    padding: 15px 17px;
                    border: 1px solid #dbe3ea;
                    background: #ffffff;
                    box-shadow:
                        0 7px 18px rgba(15, 23, 42, 0.07);
                }

                .kwb-card-header {
                    display: flex;
                    justify-content: space-between;
                    align-items: center;
                    gap: 12px;
                    margin-bottom: 9px;
                }

                .kwb-card-identity {
                    display: flex;
                    align-items: center;
                    gap: 9px;
                    min-width: 0;
                }

                .kwb-avatar {
                    display: inline-grid;
                    place-items: center;
                    width: 30px;
                    height: 30px;
                    flex: 0 0 30px;
                    border-radius: 9px;
                    font-size: 15px;
                }

                .kwb-user-avatar {
                    background: #dbeafe;
                    color: #1d4ed8;
                }

                .kwb-model-avatar {
                    background: #dcfce7;
                    color: #15803d;
                }

                .kwb-role {
                    color: var(--kwb-text);
                    font-size: 13px;
                    font-weight: 750;
                }

                .kwb-time {
                    margin-top: 1px;
                    color: #8a98aa;
                    font-size: 10px;
                }

                .kwb-chip {
                    display: inline-block;
                    margin: 0 6px 8px 0;
                    padding: 3px 8px;
                    border-radius: 999px;
                    background: #dbeafe;
                    color: #1e40af;
                    font-size: 11px;
                    font-weight: 650;
                }

                .kwb-user-body {
                    white-space: pre-wrap;
                    overflow-wrap: anywhere;
                    color: #26354a;
                    font-family:
                        ui-monospace,
                        SFMono-Regular,
                        Menlo,
                        Monaco,
                        Consolas,
                        monospace;
                    font-size: 12.5px;
                    line-height: 1.55;
                }

                .kwb-model-body {
                    overflow-wrap: anywhere;
                    color: #263142;
                    font-size: 13.5px;
                    line-height: 1.68;
                }

                .kwb-model-body h1,
                .kwb-model-body h2,
                .kwb-model-body h3,
                .kwb-model-body h4 {
                    margin-top: 1.25em;
                    margin-bottom: 0.55em;
                    color: #172033;
                    line-height: 1.3;
                }

                .kwb-model-body h1 {
                    font-size: 20px;
                }

                .kwb-model-body h2 {
                    font-size: 17px;
                }

                .kwb-model-body h3 {
                    font-size: 15px;
                }

                .kwb-model-body p {
                    margin: 0.65em 0;
                }

                .kwb-model-body ul,
                .kwb-model-body ol {
                    padding-left: 22px;
                }

                .kwb-model-body blockquote {
                    margin: 12px 0;
                    padding: 8px 13px;
                    border-left: 3px solid #8b8cec;
                    background: #f7f7ff;
                    color: #525f73;
                }

                .kwb-chat-card pre {
                    overflow-x: auto;
                    margin: 11px 0;
                    padding: 14px;
                    border: 1px solid #273449;
                    border-radius: 10px;
                    background: var(--kwb-code-bg);
                    color: #d8e1ee;
                    font-family:
                        ui-monospace,
                        SFMono-Regular,
                        Menlo,
                        Monaco,
                        Consolas,
                        monospace;
                    font-size: 12.5px;
                    line-height: 1.55;
                    box-shadow:
                        inset 0 1px 0 rgba(255, 255, 255, 0.04);
                }

                .kwb-chat-card code {
                    padding: 2px 5px;
                    border-radius: 4px;
                    background: #eef1f5;
                    color: #b42347;
                    font-family:
                        ui-monospace,
                        SFMono-Regular,
                        Menlo,
                        Monaco,
                        Consolas,
                        monospace;
                    font-size: 0.92em;
                }

                .kwb-chat-card pre code {
                    padding: 0;
                    background: transparent;
                    color: inherit;
                }

                .kwb-chat-card table {
                    width: 100%;
                    margin: 12px 0;
                    border-collapse: collapse;
                    border: 1px solid #dbe3ea;
                }

                .kwb-chat-card th,
                .kwb-chat-card td {
                    padding: 8px 10px;
                    border: 1px solid #dbe3ea;
                    text-align: left;
                }

                .kwb-chat-card th {
                    background: #f1f5f9;
                    font-weight: 700;
                }

                .kwb-copy-button {
                    flex: 0 0 auto;
                    padding: 6px 9px;
                    border: 1px solid #d8e0e9;
                    border-radius: 7px;
                    background: #f8fafc;
                    color: #475569;
                    cursor: pointer;
                    font-size: 10.5px;
                    font-weight: 700;
                    transition:
                        background 130ms ease,
                        border-color 130ms ease,
                        color 130ms ease;
                }

                .kwb-copy-button:hover {
                    border-color: #9faee8;
                    background: #eff0ff;
                    color: #4445b9;
                }

                .kwb-empty {
                    padding: 55px 20px;
                    text-align: center;
                    color: #7a8799;
                }

                .kwb-empty-icon {
                    display: grid;
                    place-items: center;
                    width: 54px;
                    height: 54px;
                    margin: 0 auto 12px;
                    border: 1px solid #dbe3ea;
                    border-radius: 16px;
                    background: #ffffff;
                    box-shadow:
                        0 8px 22px rgba(15, 23, 42, 0.07);
                    font-size: 25px;
                }

                .kwb-empty-title {
                    color: #334155;
                    font-size: 14px;
                    font-weight: 750;
                }

                .kwb-empty-copy {
                    margin-top: 5px;
                    font-size: 12px;
                }

                @media (max-width: 780px) {
                    .kwb-header {
                        padding: 19px;
                    }

                    .kwb-user-card,
                    .kwb-model-card {
                        margin-left: 0;
                        margin-right: 0;
                    }

                    .kwb-status-wrap {
                        width: 100%;
                        margin-left: 0;
                    }
                }

                @media (prefers-color-scheme: dark) {
                    :root {
                        --kwb-surface: #182235;
                        --kwb-surface-soft: #202c40;
                        --kwb-border: #334155;
                        --kwb-border-strong: #475569;
                        --kwb-text: #e5ebf4;
                        --kwb-muted: #9eacbd;
                        --kwb-shadow:
                            0 12px 30px rgba(0, 0, 0, 0.22);
                    }

                    .kwb-prompt textarea {
                        border-color: #42516a !important;
                        background: #131d2e !important;
                        color: #e5ebf4 !important;
                    }

                    .kwb-chat-shell {
                        background:
                            linear-gradient(180deg, #131d2e, #101827);
                    }

                    .kwb-user-card {
                        border-color: #2e4e7e;
                        background:
                            linear-gradient(135deg, #182a45, #172235);
                    }

                    .kwb-model-card {
                        border-color: #344257;
                        background: #182235;
                    }

                    .kwb-role,
                    .kwb-model-body,
                    .kwb-model-body h1,
                    .kwb-model-body h2,
                    .kwb-model-body h3,
                    .kwb-model-body h4,
                    .kwb-user-body {
                        color: #e2e8f0;
                    }

                    .kwb-chat-card code {
                        background: #29364a;
                        color: #ff91ad;
                    }

                    .kwb-model-body blockquote {
                        background: #202b42;
                        color: #b7c2d1;
                    }

                    .kwb-chat-card th {
                        background: #202c40;
                    }

                    .kwb-chat-card table,
                    .kwb-chat-card th,
                    .kwb-chat-card td {
                        border-color: #3b4a60;
                    }

                    .kwb-copy-button {
                        border-color: #42516a;
                        background: #202c40;
                        color: #cbd5e1;
                    }

                    .kwb-empty-icon {
                        border-color: #334155;
                        background: #182235;
                    }

                    .kwb-empty-title {
                        color: #dbe4ef;
                    }

                    .kwb-generated-files {
                        border-color: #344f76;
                        background: #182a45;
                        color: #cbd5e1;
                    }
                }
            </style>
            """
        )

        safe_work_dir = html.escape(str(self.work_dir))
        model_count = len(self.available_models)
        markdown_badge = "On" if HAS_MARKDOWN else "Fallback"

        self.header = widgets.HTML(
            value=f"""
            <div class="kwb-header">
                <div class="kwb-header-row">
                    <div class="kwb-brand">
                        <div class="kwb-logo">✦</div>
                        <div>
                            <div class="kwb-title">
                                Kaggle LLM Workbench
                            </div>
                            <div class="kwb-subtitle">
                                Prompt, inspect, export and execute generated
                                code from one notebook workspace.
                            </div>
                        </div>
                    </div>

                    <div class="kwb-header-badges">
                        <span class="kwb-header-badge">
                            ◉ {model_count} models
                        </span>
                        <span class="kwb-header-badge">
                            📁 {safe_work_dir}
                        </span>
                        <span class="kwb-header-badge">
                            Markdown: {markdown_badge}
                        </span>
                    </div>
                </div>
            </div>
            """
        )

        # Model selector.
        self.model_dropdown = widgets.Dropdown(
            options=self.available_models,
            value=self.default_model,
            description="Model",
            style={"description_width": "55px"},
            layout=widgets.Layout(
                width="470px",
                max_width="100%",
                min_width="280px",
            ),
        )

        # Temperature.
        self.temp_slider = widgets.FloatSlider(
            value=0.2,
            min=0.0,
            max=1.5,
            step=0.05,
            description="Temperature",
            readout_format=".2f",
            continuous_update=False,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="330px", max_width="100%"),
        )

        # Conversation memory.
        self.max_turns_slider = widgets.IntSlider(
            value=10,
            min=1,
            max=50,
            step=1,
            description="Memory turns",
            continuous_update=False,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="330px", max_width="100%"),
        )

        # System prompt.
        self.sys_prompt_input = widgets.Textarea(
            value=(
                "You are an elite coding assistant and software architect. "
                "Analyze, review, and optimize code provided by the user. "
                "Maintain full context across the conversation."
            ),
            layout=widgets.Layout(width="100%", height="90px"),
        )
        self.sys_prompt_input.add_class("kwb-system-prompt")

        settings_controls = widgets.HBox(
            [self.temp_slider, self.max_turns_slider],
            layout=widgets.Layout(width="100%", flex_flow="row wrap"),
        )
        settings_controls.add_class("kwb-toolbar")

        settings_content = widgets.VBox(
            [
                settings_controls,
                widgets.HTML(
                    value="""
                    <div style="
                        margin-top:6px;
                        color:var(--kwb-muted);
                        font-size:10.5px;
                        font-weight:750;
                        letter-spacing:.07em;
                        text-transform:uppercase;
                    ">
                        System instruction
                    </div>
                    """
                ),
                self.sys_prompt_input,
            ],
            layout=widgets.Layout(width="100%"),
        )
        settings_content.add_class("kwb-settings-content")

        self.settings_accordion = widgets.Accordion(
            children=[settings_content],
            layout=widgets.Layout(width="100%"),
        )
        self.settings_accordion.set_title(
            0,
            "⚙ Advanced configuration and conversation memory",
        )
        self.settings_accordion.selected_index = None

        # Main input.
        self.input_header = widgets.HTML(
            value="""
            <div style="
                display:flex;
                justify-content:space-between;
                align-items:center;
                gap:12px;
                margin-bottom:9px;
                flex-wrap:wrap;
            ">
                <div>
                    <div class="kwb-section-title">
                        New prompt
                    </div>
                    <div class="kwb-section-subtitle">
                        Paste code, request a review, or ask for a complete
                        implementation.
                    </div>
                </div>

                <div style="
                    color:var(--kwb-muted);
                    font-size:10px;
                    font-weight:650;
                ">
                    Supports fenced code and %%writefile exports
                </div>
            </div>
            """
        )

        self.textarea = widgets.Textarea(
            value="",
            placeholder=(
                "Ask a question or paste code here...\n\n"
                "Tip: Use '%%writefile filename.py' inside fenced code "
                "blocks to auto-name exported files."
            ),
            layout=widgets.Layout(width="100%", height="165px"),
        )
        self.textarea.add_class("kwb-prompt")

        # Action buttons.
        self.btn_send = widgets.Button(
            description="Send Prompt",
            button_style="primary",
            icon="paper-plane",
            tooltip="Send the prompt to the selected model",
            layout=widgets.Layout(width="150px", height="38px"),
        )

        self.btn_undo = widgets.Button(
            description="Undo Last",
            button_style="info",
            icon="undo",
            tooltip="Remove the latest conversation turn",
            layout=widgets.Layout(width="125px", height="38px"),
            disabled=True,
        )

        self.btn_save = widgets.Button(
            description="Export Files",
            button_style="success",
            icon="download",
            tooltip="Export conversation JSON and extracted code",
            layout=widgets.Layout(width="145px", height="38px"),
        )

        self.btn_reset = widgets.Button(
            description="Clear All",
            button_style="danger",
            icon="trash",
            tooltip="Clear the conversation",
            layout=widgets.Layout(width="115px", height="38px"),
        )

        self.status_label = widgets.HTML(
            value=(
                "<div class='kwb-status' style='color:#64748b;'>"
                "● Ready for a prompt"
                "</div>"
            ),
            layout=widgets.Layout(min_width="230px"),
        )
        self.status_label.add_class("kwb-status-wrap")

        # Generated file links.
        self.download_links_box = widgets.VBox(
            layout=widgets.Layout(width="100%", margin="0"),
        )

        # Execution buttons.
        self.exec_buttons_box = widgets.HBox(
            layout=widgets.Layout(
                width="100%",
                margin="0",
                flex_wrap="wrap",
            ),
        )
        self.exec_buttons_box.add_class("kwb-toolbar")

        # Execution console.
        self.exec_output = widgets.Output(
            layout=widgets.Layout(
                width="100%",
                max_height="390px",
                overflow_y="auto",
                padding="12px",
            ),
        )
        self.exec_output.add_class("kwb-console-output")

        self.btn_feed_output = widgets.Button(
            description="Send Output/Error to AI",
            icon="reply",
            button_style="warning",
            tooltip="Append the latest execution log to the prompt",
            layout=widgets.Layout(width="220px", height="34px"),
        )

        self.btn_clear_console = widgets.Button(
            description="Hide Console",
            icon="eye-slash",
            tooltip="Hide the execution console",
            layout=widgets.Layout(width="135px", height="34px"),
        )

        console_actions = widgets.HBox(
            [self.btn_feed_output, self.btn_clear_console],
            layout=widgets.Layout(margin="4px 0 0 0", flex_wrap="wrap"),
        )
        console_actions.add_class("kwb-toolbar")

        self.exec_container = widgets.VBox(
            [
                widgets.HTML(
                    value="""
                    <div class="kwb-console-title">
                        ⚡ Execution console
                    </div>
                    """
                ),
                self.exec_output,
                console_actions,
            ],
            layout=widgets.Layout(display="none", width="100%"),
        )
        self.exec_container.add_class("kwb-console")

        # Chat.
        self.chat_display = widgets.HTML(
            value=self._get_empty_chat_html()
        )

        self.chat_container = widgets.Box(
            [self.chat_display],
            layout=widgets.Layout(
                width="100%",
                max_height="680px",
                overflow_y="auto",
                padding="13px",
            ),
        )
        self.chat_container.add_class("kwb-chat-shell")
        self.chat_container.add_class("kwb-chat-scroll")

        # File attachments.
        self._init_attachment_ui()

    def _bind_events(self):
        self.btn_send.on_click(self.on_send_clicked)
        self.btn_undo.on_click(self.on_undo_clicked)
        self.btn_save.on_click(self.on_save_clicked)
        self.btn_reset.on_click(self.on_reset_clicked)
        self.btn_feed_output.on_click(self._on_feed_output_clicked)
        self.btn_clear_console.on_click(
            lambda _: setattr(
                self.exec_container.layout,
                "display",
                "none",
            )
        )

        self._bind_attachment_events()

    # ==========================================
    # 2. CORE LOGIC AND PROMPTING
    # ==========================================
    def _build_context(self, new_user_input: str, new_attachments: list) -> str:
        sys_prompt = self.sys_prompt_input.value.strip()

        context_parts = (
            [f"System:\n{sys_prompt}\n"]
            if sys_prompt
            else []
        )

        max_messages = self.max_turns_slider.value * 2

        recent_history = (
            self.chat_history[-max_messages:]
            if max_messages > 0
            else self.chat_history
        )

        for message in recent_history:
            if message["role"] == "user":
                role_string = "User"
                body = self._compose_prompt(
                    message["text"],
                    message.get("attachments", []),
                )
            else:
                role_string = "Assistant"
                body = message["text"]

            context_parts.append(f"{role_string}:\n{body}\n")

        final_prompt = self._compose_prompt(new_user_input, new_attachments)
        context_parts.append(f"User:\n{final_prompt}\n\nAssistant:")

        return "\n".join(context_parts)

    def on_send_clicked(self, _):
        typed_text = self.textarea.value.strip()
        attachments = list(self.attachments)

        if not typed_text and not attachments:
            self._set_status(
                "⚠ Please enter a prompt or attach a file first.",
                "#d93025",
            )
            return

        self._toggle_controls(disabled=True)

        chosen_model = self.model_dropdown.value
        safe_model = html.escape(str(chosen_model))
        temperature_text = f"{self.temp_slider.value:.2f}"

        self._set_status(
            f"⏳ Querying <b>{safe_model}</b> at temperature "
            f"{temperature_text}...",
            "#d97706",
        )

        try:
            context = self._build_context(typed_text, attachments)

            llm = kbench.llms[chosen_model]
            raw_response = llm.prompt(
                context,
                temperature=self.temp_slider.value,
            )
            response_text = str(raw_response)

            timestamp = datetime.datetime.now().strftime("%H:%M:%S")

            self.chat_history.append(
                {
                    "role": "user",
                    "text": typed_text,
                    "attachments": attachments,
                    "time": timestamp,
                }
            )

            self.chat_history.append(
                {
                    "role": "model",
                    "text": response_text,
                    "model_name": chosen_model,
                    "time": timestamp,
                }
            )

            self.textarea.value = ""
            self.attachments = []
            self._refresh_attachment_view()

            self.refresh_chat_view()
            self.save_chat_to_disk()

            self._set_status(
                f"✓ Response received at {timestamp} and auto-saved.",
                "#16805a",
            )

        except Exception as exc:
            self._set_status(
                f"✕ Error: {html.escape(str(exc))}",
                "#c62828",
            )

        finally:
            self._toggle_controls(disabled=False)

    def on_undo_clicked(self, _):
        if len(self.chat_history) >= 2:
            self.chat_history.pop()
            last_user_message = self.chat_history.pop()

            self.textarea.value = last_user_message["text"]
            self.attachments = list(last_user_message.get("attachments", []))
            self._refresh_attachment_view()

            self.refresh_chat_view()

            self._set_status(
                "↩ Last turn removed and restored to the prompt box.",
                "#2563b8",
            )

        self.btn_undo.disabled = len(self.chat_history) < 2

    def on_save_clicked(self, _):
        if not self.chat_history:
            self._set_status(
                "⚠ Nothing is available to export yet.",
                "#d97706",
            )
            return

        self.save_chat_to_disk()

        safe_directory = html.escape(str(self.work_dir))

        self._set_status(
            f"💾 Files exported to {safe_directory}.",
            "#2563b8",
        )

    def on_reset_clicked(self, _):
        self.chat_history.clear()
        self._last_exec_log = ""
        self.attachments = []
        self._refresh_attachment_view()

        self.refresh_chat_view()

        self.download_links_box.children = []
        self.exec_buttons_box.children = []
        self.exec_container.layout.display = "none"
        self.exec_output.clear_output(wait=True)

        self.btn_undo.disabled = True

        self._set_status("Chat history cleared.", "#64748b")

    # ==========================================
    # 3. DISK EXPORT, EXTRACTION AND EXECUTION
    # ==========================================
    def save_chat_to_disk(self):
        if not self.chat_history:
            return

        last_answer = (
            self.chat_history[-1]["text"]
            if self.chat_history[-1]["role"] == "model"
            else ""
        )

        # Matches fenced blocks with optional language tag.
        # Supports both \n and \r\n line endings.
        code_blocks = re.findall(
            rf"{CODE_FENCE}([a-zA-Z0-9_+\-]*)[ \t]*\r?\n(.*?){CODE_FENCE}",
            last_answer,
            flags=re.DOTALL,
        )

        export_payload = {
            "timestamp": datetime.datetime.now().isoformat(),
            "model": self.model_dropdown.value,
            "system_prompt": self.sys_prompt_input.value,
            "temperature": self.temp_slider.value,
            "last_response": last_answer,
            "extracted_code_blocks": [
                {
                    "lang": language or "txt",
                    "code": code,
                }
                for language, code in code_blocks
            ],
            "full_history": self.chat_history,
        }

        json_path = self.work_dir / "argon_output.json"

        with open(json_path, "w", encoding="utf-8") as output_file:
            json.dump(
                export_payload,
                output_file,
                ensure_ascii=False,
                indent=2,
            )

        extension_map = {
            "python": "py",
            "py": "py",
            "javascript": "js",
            "js": "js",
            "typescript": "ts",
            "ts": "ts",
            "html": "html",
            "css": "css",
            "bash": "sh",
            "shell": "sh",
            "sh": "sh",
            "sql": "sql",
            "json": "json",
            "yaml": "yaml",
            "yml": "yml",
            "cpp": "cpp",
            "c++": "cpp",
            "c": "c",
            "java": "java",
            "text": "txt",
            "txt": "txt",
        }

        saved_files = ["argon_output.json"]

        for index, (language, block) in enumerate(code_blocks):
            lines = block.strip().splitlines()

            normalized_language = language.lower().strip()
            extension = extension_map.get(normalized_language, "txt")

            if lines and lines[0].strip().startswith("%%writefile"):
                raw_filename = (
                    lines[0]
                    .replace("%%writefile", "", 1)
                    .strip()
                )

                safe_filename = os.path.basename(raw_filename)

                if not safe_filename:
                    safe_filename = f"extracted_{index + 1}.{extension}"

                content = "\n".join(lines[1:])

            else:
                safe_filename = f"argon_code_{index + 1}.{extension}"
                content = block

            file_path = self.work_dir / safe_filename

            with open(file_path, "w", encoding="utf-8") as output_file:
                output_file.write(content)

            saved_files.append(safe_filename)

        links = []

        for filename in saved_files:
            safe_href = html.escape(filename, quote=True)
            safe_label = html.escape(filename)

            links.append(
                f"<a href='{safe_href}' target='_blank'>"
                f"📄 {safe_label}</a>"
            )

        links_html = (
            " <span style='opacity:.35;'>•</span> "
        ).join(links)

        self.download_links_box.children = [
            widgets.HTML(
                value=(
                    "<div class='kwb-generated-files'>"
                    "<strong>Generated files</strong>"
                    "<div style='margin-top:3px;'>"
                    f"{links_html}"
                    "</div>"
                    "</div>"
                )
            )
        ]

        # Create execution buttons for Python and Bash files.
        run_buttons = []

        for filename in saved_files:
            if filename.endswith((".py", ".sh")):
                run_button = widgets.Button(
                    description=f"Run {filename}",
                    button_style="success",
                    icon="play",
                    tooltip=(
                        f"Execute {filename} in the current "
                        "notebook session"
                    ),
                    layout=widgets.Layout(width="auto", height="34px"),
                )

                run_button.on_click(
                    lambda _, selected_file=filename:
                    self.execute_file(selected_file)
                )

                run_buttons.append(run_button)

        self.exec_buttons_box.children = run_buttons

        # Keep the workspace dropdown in sync with new exports.
        self.attach_select.options = self._list_workspace_files()

    def execute_file(self, filename: str):
        file_path = self.work_dir / filename

        if not file_path.exists():
            self._set_status(
                f"⚠ File {html.escape(filename)} was not found.",
                "#d93025",
            )
            return

        code = file_path.read_text(encoding="utf-8")

        self.exec_container.layout.display = "flex"
        self.exec_output.clear_output(wait=True)

        safe_filename = html.escape(filename)

        self._set_status(
            f"⚡ Executing <b>{safe_filename}</b>...",
            "#d97706",
        )

        ipython = get_ipython()
        execution_successful = True
        captured = None

        with self.exec_output:
            print(f"┌─ Executing: {filename}")
            print(f"└─ Path: {file_path}\n")

            with capture_output(
                stdout=True,
                stderr=True,
                display=True,
            ) as captured:
                try:
                    if filename.endswith(".sh"):
                        if ipython is None:
                            raise RuntimeError(
                                "A live IPython session is required "
                                "to execute shell scripts."
                            )

                        quoted_path = shlex.quote(str(file_path))
                        result = ipython.system(f"bash {quoted_path}")

                        if isinstance(result, int) and result != 0:
                            execution_successful = False

                    else:
                        if ipython is not None:
                            result = ipython.run_cell(code)

                            if result.error_in_exec is not None:
                                execution_successful = False

                            if result.error_before_exec is not None:
                                execution_successful = False

                        else:
                            execution_globals = {
                                "__name__": "__main__",
                                "__file__": str(file_path),
                            }

                            exec(
                                compile(code, str(file_path), "exec"),
                                execution_globals,
                                execution_globals,
                            )

                except Exception:
                    traceback.print_exc()
                    execution_successful = False

            # Re-display captured text, tables and plots.
            captured.show()

        log_parts = []

        if captured is not None and captured.stdout:
            log_parts.append("STDOUT:\n" f"{captured.stdout.strip()}")

        if captured is not None and captured.stderr:
            log_parts.append("STDERR / TRACEBACK:\n" f"{captured.stderr.strip()}")

        if log_parts:
            log_string = "\n\n".join(log_parts)
        else:
            log_string = "Code executed silently with no stdout/stderr output."

        self._last_exec_log = (
            f"I executed `{filename}` and got the following output:\n"
            f"{CODE_FENCE}text\n{log_string[-4000:]}\n{CODE_FENCE}"
        )

        if execution_successful:
            self._set_status(
                f"✓ Execution of <b>{safe_filename}</b> finished.",
                "#16805a",
            )
        else:
            self._set_status(
                f"✕ Execution failed in <b>{safe_filename}</b>. "
                "Use “Send Output/Error to AI” to debug it.",
                "#c62828",
            )

    def _on_feed_output_clicked(self, _):
        if not self._last_exec_log:
            self._set_status(
                "⚠ No execution output is available yet.",
                "#d97706",
            )
            return

        current_text = self.textarea.value.strip()

        prefix = f"{current_text}\n\n" if current_text else ""

        self.textarea.value = (
            f"{prefix}"
            f"{self._last_exec_log}\n\n"
            "Please analyze the result or fix any errors:"
        )

        self._set_status(
            "Execution log appended to the prompt box.",
            "#2563b8",
        )

    # ==========================================
    # 3B. FILE ATTACHMENTS
    # ==========================================
    def _init_attachment_ui(self):
        self.attachments = []

        self.attach_select = widgets.Dropdown(
            options=self._list_workspace_files(),
            description="Workspace",
            style={"description_width": "75px"},
            layout=widgets.Layout(width="340px", max_width="100%"),
        )

        self.btn_refresh_files = widgets.Button(
            description="Refresh",
            icon="refresh",
            tooltip="Rescan the working directory",
            layout=widgets.Layout(width="105px", height="34px"),
        )

        self.btn_attach_existing = widgets.Button(
            description="Attach file",
            button_style="info",
            icon="paperclip",
            tooltip="Attach the selected workspace file",
            layout=widgets.Layout(width="130px", height="34px"),
        )

        self.upload_widget = widgets.FileUpload(
            accept=",".join(TEXT_LANG_BY_EXT.keys()),
            multiple=True,
            description="Upload",
            layout=widgets.Layout(width="auto"),
        )

        self.btn_attach_upload = widgets.Button(
            description="Attach uploads",
            button_style="info",
            icon="upload",
            tooltip="Attach the uploaded files",
            layout=widgets.Layout(width="140px", height="34px"),
        )

        self.attachment_list = widgets.HTML()

        self.btn_clear_attachments = widgets.Button(
            description="Clear attachments",
            button_style="danger",
            icon="times",
            layout=widgets.Layout(width="150px", height="34px"),
            disabled=True,
        )

        self.attachment_panel = widgets.VBox(
            [
                widgets.HTML(
                    value=(
                        "<div class='kwb-section-title'>Attach files</div>"
                        "<div class='kwb-section-subtitle'>"
                        "Attached files are sent with your next prompt."
                        "</div>"
                    )
                ),
                widgets.HBox(
                    [
                        self.attach_select,
                        self.btn_refresh_files,
                        self.btn_attach_existing,
                    ],
                    layout=widgets.Layout(flex_flow="row wrap", gap="8px"),
                ),
                widgets.HBox(
                    [self.upload_widget, self.btn_attach_upload],
                    layout=widgets.Layout(flex_flow="row wrap", gap="8px"),
                ),
                self.attachment_list,
                self.btn_clear_attachments,
            ],
            layout=widgets.Layout(width="100%", gap="8px"),
        )
        self.attachment_panel.add_class("kwb-panel")

        self._refresh_attachment_view()

    def _bind_attachment_events(self):
        self.btn_refresh_files.on_click(self.on_refresh_files_clicked)
        self.btn_attach_existing.on_click(self.on_attach_existing_clicked)
        self.btn_attach_upload.on_click(self.on_attach_upload_clicked)
        self.btn_clear_attachments.on_click(self.on_clear_attachments_clicked)

    def _list_workspace_files(self) -> list:
        return [
            path.name
            for path in sorted(self.work_dir.iterdir())
            if path.is_file()
            and path.suffix.lower() in TEXT_LANG_BY_EXT
        ]

    def _decode_text(self, data: bytes, name: str) -> str:
        if len(data) > ATTACH_MAX_BYTES:
            raise ValueError(
                f"{name} is {len(data):,} bytes. "
                f"The limit is {ATTACH_MAX_BYTES:,} bytes."
            )

        if b"\x00" in data:
            raise ValueError(f"{name} appears to be binary.")

        try:
            return data.decode("utf-8")
        except UnicodeDecodeError as exc:
            raise ValueError(f"{name} is not valid UTF-8 text.") from exc

    def _read_workspace_file(self, filename: str) -> str:
        # Path(filename).name strips any directory components.
        path = self.work_dir / Path(filename).name

        if not path.is_file():
            raise ValueError(f"{filename} was not found in the workspace.")

        return self._decode_text(path.read_bytes(), path.name)

    @staticmethod
    def _normalize_uploads(value) -> list:
        # ipywidgets 8 returns a tuple of dicts.
        # ipywidgets 7 returns a dict keyed by filename.
        if isinstance(value, dict):
            return [
                (name, bytes(info["content"]))
                for name, info in value.items()
            ]

        return [
            (info["name"], bytes(info["content"]))
            for info in value
        ]

    def _add_attachment(self, name: str, content: str, source: str):
        self.attachments = [
            item for item in self.attachments if item["name"] != name
        ]

        if len(self.attachments) >= ATTACH_MAX_FILES:
            raise ValueError(
                f"You can attach up to {ATTACH_MAX_FILES} files per prompt."
            )

        self.attachments.append(
            {
                "name": name,
                "content": content,
                "source": source,
                "lang": TEXT_LANG_BY_EXT.get(
                    Path(name).suffix.lower(),
                    "text",
                ),
            }
        )

    @staticmethod
    def _fence_for(content: str) -> str:
        # Use a fence longer than any backtick run in the content.
        longest = max(
            (len(run) for run in re.findall(r"`+", content)),
            default=0,
        )
        return "`" * max(3, longest + 1)

    def _compose_prompt(self, text: str, attachments: list) -> str:
        if not attachments:
            return text

        blocks = []

        for item in attachments:
            fence = self._fence_for(item["content"])
            blocks.append(
                f"Attached file: {item['name']}\n"
                f"{fence}{item['lang']}\n"
                f"{item['content']}\n"
                f"{fence}"
            )

        instruction = text or "Please analyze the attached file(s)."
        return "\n\n".join(blocks) + "\n\n" + instruction

    def _refresh_attachment_view(self):
        if not self.attachments:
            self.attachment_list.value = (
                "<div style='color:#7a8799; font-size:12px;'>"
                "No files attached to the next prompt."
                "</div>"
            )
            self.btn_clear_attachments.disabled = True
            return

        chips = []

        for item in self.attachments:
            line_count = len(item["content"].splitlines())
            chips.append(
                "<span class='kwb-chip'>"
                f"📎 {html.escape(item['name'])}"
                f" <span style='opacity:.7;'>({line_count:,} lines)</span>"
                "</span>"
            )

        self.attachment_list.value = "<div>" + "".join(chips) + "</div>"
        self.btn_clear_attachments.disabled = False

    def on_refresh_files_clicked(self, _):
        files = self._list_workspace_files()
        self.attach_select.options = files
        self.attach_select.value = files[0] if files else None

        self._set_status(
            f"🔄 Found {len(files)} text file(s) in the workspace.",
            "#2563b8",
        )

    def on_attach_existing_clicked(self, _):
        filename = self.attach_select.value

        if not filename:
            self._set_status("⚠ No workspace file is selected.", "#d97706")
            return

        try:
            content = self._read_workspace_file(filename)
            self._add_attachment(filename, content, "workspace")
        except ValueError as exc:
            self._set_status(f"✕ {html.escape(str(exc))}", "#c62828")
            return

        self._refresh_attachment_view()
        self._set_status(
            f"📎 Attached {html.escape(filename)}.",
            "#16805a",
        )

    def on_attach_upload_clicked(self, _):
        uploads = self._normalize_uploads(self.upload_widget.value)

        if not uploads:
            self._set_status(
                "⚠ Choose one or more files to upload first.",
                "#d97706",
            )
            return

        attached, errors = [], []

        for name, data in uploads:
            try:
                content = self._decode_text(data, name)
                self._add_attachment(name, content, "upload")
                attached.append(name)
            except ValueError as exc:
                errors.append(str(exc))

        # Reset to an empty value of the same container type
        # (tuple in ipywidgets 8, dict in ipywidgets 7).
        self.upload_widget.value = type(self.upload_widget.value)()
        self._refresh_attachment_view()

        if errors:
            self._set_status(
                f"✕ Skipped: {html.escape(' | '.join(errors))}",
                "#c62828",
            )
        else:
            self._set_status(
                f"📎 Attached {len(attached)} uploaded file(s).",
                "#16805a",
            )

    def on_clear_attachments_clicked(self, _):
        self.attachments = []
        self._refresh_attachment_view()
        self._set_status("Attachments cleared.", "#64748b")

    # ==========================================
    # 4. VIEW RENDERING AND HELPERS
    # ==========================================
    def _get_copy_js(
        self,
        element_id: str,
        btn_text: str = "Copied!",
    ) -> str:
        return (
            f"var t=document.getElementById('{element_id}');"
            "var btn=this;"
            "var original=btn.innerHTML;"
            "var done=function(){"
            f"btn.innerHTML='✓ {btn_text}';"
            "btn.style.background='#dcfce7';"
            "btn.style.color='#15803d';"
            "btn.style.borderColor='#86efac';"
            "setTimeout(function(){"
            "btn.innerHTML=original;"
            "btn.style.background='';"
            "btn.style.color='';"
            "btn.style.borderColor='';"
            "},1800);"
            "};"
            "if(navigator.clipboard && window.isSecureContext){"
            "navigator.clipboard.writeText(t.value).then(done);"
            "}else{"
            "t.style.display='block';"
            "t.select();"
            "document.execCommand('copy');"
            "t.style.display='none';"
            "done();"
            "}"
        )

    def _format_model_text(self, text: str, msg_idx: int) -> str:
        del msg_idx  # Retained for signature compatibility.

        if HAS_MARKDOWN:
            # Markdown handles escaping inside code blocks itself.
            return markdown.markdown(
                text,
                extensions=["fenced_code", "tables"],
            )

        return (
            "<div style='"
            "white-space:pre-wrap;"
            "font-family:ui-monospace, SFMono-Regular, "
            "Menlo, Monaco, Consolas, monospace;"
            "'>"
            f"{html.escape(text)}"
            "</div>"
        )

    def refresh_chat_view(self):
        if not self.chat_history:
            self.chat_display.value = self._get_empty_chat_html()
            return

        cards = []

        for index, entry in enumerate(self.chat_history):
            time_string = html.escape(entry.get("time", ""))

            if entry["role"] == "user":
                safe_user_text = html.escape(entry["text"])

                chips = "".join(
                    "<span class='kwb-chip'>"
                    f"📎 {html.escape(item['name'])}</span>"
                    for item in entry.get("attachments", [])
                )

                cards.append(
                    f"""
                    <div class="kwb-chat-card kwb-user-card">
                        <div class="kwb-card-header">
                            <div class="kwb-card-identity">
                                <div class="kwb-avatar kwb-user-avatar">◉</div>
                                <div>
                                    <div class="kwb-role">You</div>
                                    <div class="kwb-time">{time_string}</div>
                                </div>
                            </div>
                        </div>

                        {chips}

                        <div class="kwb-user-body">{safe_user_text}</div>
                    </div>
                    """
                )

            else:
                raw_escaped = html.escape(entry["text"])

                rendered_body = self._format_model_text(
                    entry["text"],
                    index,
                )

                selected_name = html.escape(
                    entry.get("model_name", "Model")
                )

                copy_javascript = self._get_copy_js(
                    f"raw_out_{index}",
                    "Copied",
                )

                cards.append(
                    f"""
                    <div class="kwb-chat-card kwb-model-card">
                        <div class="kwb-card-header">
                            <div class="kwb-card-identity">
                                <div class="kwb-avatar kwb-model-avatar">✦</div>

                                <div style="min-width:0;">
                                    <div class="kwb-role">{selected_name}</div>
                                    <div class="kwb-time">{time_string}</div>
                                </div>
                            </div>

                            <button
                                class="kwb-copy-button"
                                onclick="{copy_javascript}"
                                title="Copy the raw model response"
                            >
                                ⧉ Copy Markdown
                            </button>
                        </div>

                        <textarea
                            id="raw_out_{index}"
                            style="display:none;"
                        >{raw_escaped}</textarea>

                        <div class="kwb-model-body">
                            {rendered_body}
                        </div>
                    </div>
                    """
                )

        self.chat_display.value = "".join(cards)

    def _get_empty_chat_html(self) -> str:
        return """
        <div class="kwb-empty">
            <div class="kwb-empty-icon">💬</div>
            <div class="kwb-empty-title">
                Your workspace is ready
            </div>
            <div class="kwb-empty-copy">
                Enter a prompt above to start a conversation.
                Generated code and exports will appear here.
            </div>
        </div>
        """

    def _set_status(
        self,
        html_msg: str,
        color: str = "#202124",
    ):
        self.status_label.value = (
            "<div class='kwb-status' "
            f"style='color:{color}; border-color:{color}33;'>"
            f"{html_msg}"
            "</div>"
        )

    def _toggle_controls(self, disabled: bool):
        self.btn_send.disabled = disabled
        self.btn_save.disabled = disabled
        self.btn_reset.disabled = disabled
        self.btn_undo.disabled = disabled or len(self.chat_history) < 2

        self.btn_attach_existing.disabled = disabled
        self.btn_attach_upload.disabled = disabled
        self.btn_refresh_files.disabled = disabled
        self.btn_clear_attachments.disabled = (
            disabled or not self.attachments
        )

        self.model_dropdown.disabled = disabled
        self.temp_slider.disabled = disabled
        self.max_turns_slider.disabled = disabled
        self.sys_prompt_input.disabled = disabled

    def display(self):
        top_bar = widgets.HBox(
            [
                widgets.HTML(
                    value="""
                    <div class="kwb-toolbar-label">
                        Active model
                    </div>
                    """
                ),
                self.model_dropdown,
            ],
            layout=widgets.Layout(
                width="100%",
                align_items="center",
                flex_flow="row wrap",
            ),
        )
        top_bar.add_class("kwb-toolbar")

        toolbar_panel = widgets.VBox(
            [top_bar],
            layout=widgets.Layout(width="100%"),
        )
        toolbar_panel.add_class("kwb-panel")

        input_panel = widgets.VBox(
            [self.input_header, self.textarea],
            layout=widgets.Layout(width="100%"),
        )
        input_panel.add_class("kwb-panel")

        action_bar = widgets.HBox(
            [
                self.btn_send,
                self.btn_undo,
                self.btn_save,
                self.btn_reset,
                self.status_label,
            ],
            layout=widgets.Layout(
                width="100%",
                align_items="center",
                flex_flow="row wrap",
            ),
        )
        action_bar.add_class("kwb-actionbar")
        action_bar.add_class("kwb-toolbar")

        files_area = widgets.VBox(
            [self.download_links_box, self.exec_buttons_box],
            layout=widgets.Layout(width="100%"),
        )

        chat_heading = widgets.HTML(
            value="""
            <div style="
                display:flex;
                justify-content:space-between;
                align-items:center;
                margin:4px 2px -3px;
                gap:12px;
                flex-wrap:wrap;
            ">
                <div class="kwb-section-title">
                    Conversation
                </div>

                <div style="
                    color:var(--kwb-muted);
                    font-size:10px;
                ">
                    Recent messages are included according to memory settings
                </div>
            </div>
            """
        )

        app_layout = widgets.VBox(
            [
                self.theme,
                self.header,
                toolbar_panel,
                self.settings_accordion,
                input_panel,
                self.attachment_panel,
                action_bar,
                files_area,
                self.exec_container,
                chat_heading,
                self.chat_container,
            ],
            layout=widgets.Layout(width="100%"),
        )

        app_layout.add_class("kwb-root")
        app_layout.add_class("kwb-shell")

        display(app_layout)


# ==========================================
# 5. LAUNCH WORKBENCH
# ==========================================
workbench = KaggleLLMWorkbench()
workbench.display()